    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.
    
    Task 5 (LS3): Implement a program which, (a) given one of the feature models and (b) a value k,
    – creates (and saves) a label-label similarity matrix,
    – performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this label-label
    similarity matrix,
    – stores the latent semantics in a properly named output file
    – lists label-weight pairs, ordered in decreasing order of weights

In [8]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [9]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " label latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 100  label latent semantics under  fc  feature space using:  svd


In [10]:
from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity
from utils.dataset_utils import initialize_dataset

label_similarity_generator = LabelLabelSimilarity(feature_vectors, initialize_dataset().categories)
label_feature_vectors = label_similarity_generator.get_matrix()

print("Label-label similarity matrix for faces as example:\n")
print(label_feature_vectors[0])
print("Shape: ", label_feature_vectors[0][1].shape)

Files already downloaded and verified
Label-label similarity matrix for faces as example:

('Faces', array([1.00000000e+00, 9.60584402e-01, 6.55803531e-02, 0.00000000e+00,
       1.54499143e-01, 0.00000000e+00, 3.31209227e-02, 1.09157033e-01,
       6.65913522e-02, 9.91684571e-02, 0.00000000e+00, 2.38687266e-02,
       0.00000000e+00, 1.93770945e-01, 0.00000000e+00, 6.85443208e-02,
       1.32241055e-01, 5.58832437e-02, 3.02741174e-02, 0.00000000e+00,
       1.44533142e-01, 2.38082007e-01, 0.00000000e+00, 1.36598218e-02,
       2.75379769e-03, 1.77518219e-01, 8.92516747e-02, 8.64985138e-02,
       0.00000000e+00, 8.83039844e-04, 1.92280278e-01, 9.32242721e-02,
       2.80332714e-01, 0.00000000e+00, 1.55025363e-01, 2.40313053e-01,
       0.00000000e+00, 0.00000000e+00, 1.41856819e-01, 0.00000000e+00,
       0.00000000e+00, 3.00697256e-02, 1.07354060e-01, 3.03628832e-01,
       0.00000000e+00, 9.88438204e-02, 3.76361534e-02, 0.00000000e+00,
       1.71870902e-01, 5.14730737e-02, 0.000000

In [11]:
# Store label-label feature vectors for use along with the reducer.

from utils.database_utils import store

store(label_feature_vectors, f'label_label_{FEATURE_SPACE}.pt')


 Saving:  label_label_fc.pt 



In [12]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(label_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(label_feature_vectors)

latent_semantics = reducer.reduce_features(label_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[-1.10639522e+00 -3.93895016e-01  4.48927914e-01 ...  7.07292828e-03
  -5.87506366e-03  1.10945994e-02]
 [-9.97975512e-01 -3.70975933e-01  3.99654879e-01 ... -6.63984988e-03
   5.99334902e-03 -1.10011817e-02]
 [-1.23946175e+00  3.66259282e-01 -4.58237037e-01 ... -1.26682418e-04
   1.07181051e-04 -6.27388265e-04]
 ...
 [-1.10379706e+00  1.08007404e-01 -2.30812828e-01 ... -6.03892386e-03
   5.20682123e-03 -2.22520786e-04]
 [-2.30904067e+00 -1.00680060e+00 -2.80271063e-01 ... -1.97695971e-03
  -1.12656323e-04 -4.84107327e-04]
 [-2.56248859e+00 -7.64410259e-01  2.43589853e-01 ... -5.00067716e-04
   1.42910957e-03 -6.47093453e-04]]
Shape:  (101, 100)


In [13]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS3_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

store(reducer, f'LS3_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS3_fc_svd_reducer.pt 



In [14]:
# List label-weight pairs, ordered in decreasing order of weights

# We are to showcase which labels contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

labels = [feature_tuple[0] for feature_tuple in label_feature_vectors.values()]

image_weight_tuples = list(zip(labels, similarity_matrix))

print("Label - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for LABEL, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(Label: ", LABEL, ", Weight: ", weight[i], end="),\t")

Label - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(Label:  car_side , Weight:  -0.012922208759228417),	(Label:  ferry , Weight:  -0.017169620543383994),	(Label:  ibis , Weight:  -0.02665104036861119),	(Label:  joshua_tree , Weight:  -0.026880534355430852),	(Label:  ketch , Weight:  -0.02996846275214408),	(Label:  emu , Weight:  -0.03464076919689637),	(Label:  schooner , Weight:  -0.035677890493769605),	(Label:  helicopter , Weight:  -0.03636295870334973),	(Label:  wheelchair , Weight:  -0.04018367705164094),	(Label:  rhino , Weight:  -0.04091843413594813),	(Label:  hawksbill , Weight:  -0.04253151322368189),	(Label:  llama , Weight:  -0.04334601458190738),	(Label:  pigeon , Weight:  -0.04357656773707959),	(Label:  Motorbikes , Weight:  -0.04475378042631114),	(Label:  kangaroo , Weight:  -0.04828273633007541),	(Label:  gerenuk , Weight:  -0.04849765219181952),	(Label:  hedgehog , Weight:  -0.049975633251524404),	(Label:  grand_piano